# Align $X$ and $Y$ 
##### This is due to the fact that they both have different sizes.

$$TR = 1.3s$$

In [ ]:
import pandas as pd
import numpy as np


Y_npy_file = "/Volumes/LaCie/EPFL/Mastersem3/SemesterProjectND/Y/sub2/After_The_Rain/Y_voxels.npy"
X_csv_file = '/Volumes/LaCie/EPFL/Mastersem3/SemesterProjectND/DataframeX/After_The_Rain_exp/Final_df.csv'
X = pd.read_csv(X_csv_file)
Y = np.load(Y_npy_file)


In [ ]:
# X feature dataframe
# Create TR intervals (1.3s each) starting from 0 seconds
TR_duration = 1.3
num_TRs = int(X['Time (s)'].max() // TR_duration)

# Assign TR indices to each time point
X['TR'] = (X['Time (s)'] // TR_duration).astype(int)

In [ ]:
# Group by TR and compute the mean for each feature
X_avg = X.groupby('TR').mean().reset_index()

# Drop the 'Time (s)' column if not needed
X_avg = X_avg.drop(columns=['Time (s)'], errors='ignore')


import numpy as np

# Select columns whose names start with 'Object', 'Action', or 'Scene'
columns_to_round = [col for col in X_avg.columns if col.startswith(('Object', 'Action', 'Scene'))]

# Apply rounding: Set all non-zero values to 1
X_avg[columns_to_round] = np.where(X_avg[columns_to_round] != 0, 1, 0)

X_avg = X_avg.drop(columns=['TR'], errors='ignore')


In [ ]:
X_avg

In [ ]:
Y.shape

#### The following needs to be rectified according to the 4TR variation (!)

In [ ]:

# TRs to remove : 90 / 1.3 = ? 69.23

TRs_to_remove = 69

#### We start by removing the washed out element from the fmri :
Y_trimmed = Y[TRs_to_remove:-TRs_to_remove]
#print(Y_trimmed.shape)

start = int(np.floor((Y_trimmed.shape[0]-X_avg.shape[0])/2))

Y_weird = Y_trimmed[start: -(start+1)]
#print(np.floor((Y_trimmed.shape[0]-X_avg.shape[0])/2))
#print(int(np.floor((Y_trimmed.shape[0]-X_avg.shape[0])/2)))
print(Y_weird.shape)

In [ ]:
import nibabel as nib
import numpy as np

# Assuming fmri_img is a NIfTI image loaded using nibabel
fmri_img = nib.load("/Volumes/LaCie/EPFL/Mastersem3/SemesterProjectND/Y/sub2/After_The_Rain/smoothed_fmri.nii.gz")
data = fmri_img.get_fdata()  # Get the image data as a NumPy array

# Define the number of TRs to remove
TRs_to_remove = 69

##### We start by removing the washed out element from the fmri : #####
# Step 1: Trim the time points (removing 69 time points from both ends)
data_trimmed = data[..., TRs_to_remove:-TRs_to_remove]  # Trim the last and first 69 time points

# Check the shape of the trimmed data
print(f"Trimmed data shape: {data_trimmed.shape}")

# Step 2: Adjust the time points to match X_avg
# Assuming X_avg.shape[0] is the number of time points you want in the trimmed data
start = int(np.floor((data_trimmed.shape[3] - X_avg.shape[0]) / 2))

# Step 3: Extract the portion of the data to match the shape of X_avg
data_weird = data_trimmed[..., start: -(start + 1)]  # Slice to align with X_avg

# Check the final shape
print(f"Final trimmed data shape: {data_weird.shape}")

affine = fmri_img.affine  # Get the affine matrix from the original image

# Step 1: Create a new NIfTI image with data_weird and the original affine matrix
new_fmri_img = nib.Nifti1Image(data_weird, affine)

### Several ways of applying GLM

#### 1) With nilearn for neuro-imaging data, might be good

In [ ]:
import numpy as np
from nilearn.glm.first_level import FirstLevelModel

# Example: Create design matrix for fMRI
# X is the design matrix (regressors)
# Y is the fMRI data (time series of voxels around the brain)
# new_fmri_img is the trimmed fmri data

# Initialize the GLM model
model = FirstLevelModel(t_r=1.3, slice_time_ref=0.5)

# Fit the model to your data
model.fit(new_fmri_img, design_matrices=X_avg)

# Get the results
#z_map = model.compute_contrast()

In [ ]:
# Create a contrast vector for comparing brightness and contrast
contrast_def = np.zeros(200)  # nb of feature columns

contrast_def[0] = 1  # Brightness 
contrast_def[1] = -1  # Contrast 

# Compute the contrast
z_map = model.compute_contrast(contrast_def)

In [ ]:
from nilearn import plotting

# Visualize the contrast result (z_map)
plotting.plot_stat_map(z_map, title="Contrast: Brightness vs. Contrast Value", display_mode='ortho', draw_cross=True, colorbar=True)

# Show the plot
plotting.show()


In [ ]:
fmri_img.shape

In [ ]:
X_avg.columns.tolist()